In [ ]:
import os
import io
import math
import base64
import requests
import concurrent.futures
from functools import partial
from PIL import Image

# ----------------------------
# CONFIG
# ----------------------------
ACCOUNT_HASH = "LJW9TBw5H2BwGc8uUBxFZA"   # <-- fill in
IMG_IDS_FILE = "cf_image_ids.txt"

TILE_W = 32
TILE_H = 32
GRID_W = 50

OUT_IMAGE = "collage.webp"
OUT_HTML = "images.html"

MAX_WORKERS = 20
# ----------------------------


# ----------------------------
# HELPERS
# ----------------------------
def image_url(image_id: str) -> str:
    return f"https://imagedelivery.net/{ACCOUNT_HASH}/{image_id}/thumbxsm"


def fetch_image_as_base64(url: str) -> str:
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    img = Image.open(io.BytesIO(response.content))

    buf = io.BytesIO()
    img.save(buf, format="WEBP", quality=60)
    webp_bytes = buf.getvalue()

    encoded = base64.b64encode(webp_bytes).decode("utf-8")
    return f"data:image/webp;base64,{encoded}"
# ----------------------------


# ----------------------------
# LOAD IMAGE IDS
# ----------------------------
with open(IMG_IDS_FILE, 'r') as r:
    image_ids = [i.strip() for i in r.readlines() if i.strip()]

total_count = len(image_ids)
rows = math.ceil(total_count / GRID_W)

print(f"Loaded {total_count} Cloudflare image IDs.")
print(f"Building {GRID_W}×{rows} grid.")


# ----------------------------
# MULTITHREADED IMAGE FETCHER
# ----------------------------
def fetch_single_image(index_imgid, total):

    """Fetch image ID → base64 → PIL (bytes decoded)."""
    index, img_id = index_imgid
    url = image_url(img_id)

    print(f"[{index+1}/{total}]")

    try:
        base64_data_url = fetch_image_as_base64(url)
        header, b64 = base64_data_url.split(",", 1)
        img_bytes = base64.b64decode(b64)
        pil = Image.open(io.BytesIO(img_bytes)).convert("RGBA")
        return pil

    except Exception as e:
        print(f"[{index+1}/{total}] FAILED {img_id}: {e}")
        return None
    


pairs = list(enumerate(image_ids))
pil_images = [None] * len(image_ids)

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for (index, img_id), img in zip(
        pairs,
        executor.map(partial(fetch_single_image, total=total_count), pairs)
    ):
        pil_images[index] = img

# Filter out failures (rare but possible)
good_images = [p for p in pil_images if p is not None]
print(f"Downloaded {len(good_images)}/{total_count} images successfully.")

if len(good_images) == 0:
    raise RuntimeError("No images downloaded — cannot build collage.")


# ----------------------------
# BUILD COLLAGE SPRITE SHEET
# ----------------------------
collage = Image.new("RGBA", (GRID_W * TILE_W, rows * TILE_H))
coords = []

idx = 0
for row in range(rows):
    for col in range(GRID_W):
        if idx >= len(good_images):
            break

        tile = good_images[idx]
        x = col * TILE_W
        y = row * TILE_H
        collage.paste(tile, (x, y))

        coords.append((image_ids[idx], x, y, x + TILE_W, y + TILE_H))
        idx += 1

collage.save(OUT_IMAGE, "WEBP", quality=80)
print("Saved merged image:", OUT_IMAGE)


# ----------------------------
# GENERATE HTML MAP
# ----------------------------
html_parts = []

html_parts.append(f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Image Map</title>
<style>
body {{
  background: #111;
  color: #ddd;
  font-family: sans-serif;
}}
</style>
</head>
<body>

<img src="{OUT_IMAGE}" usemap="#imgmap" draggable="false">

<map name="imgmap">
""")

for img_id, x1, y1, x2, y2 in coords:
    html_parts.append(
        f'  <area shape="rect" coords="{x1},{y1},{x2},{y2}" href="https://imagedelivery.net/LJW9TBw5H2BwGc8uUBxFZA/${img_id}/public" />'
    )

html_parts.append("</map>\n</body>\n</html>")

with open(OUT_HTML, "w") as w:
    w.write("\n".join(html_parts))

print("Saved HTML:", OUT_HTML)
